# DenseNet121 Glaucoma Detection Using ACRIMA Fundus Images

**Project:** Comparative Analysis of Deep Learning Architectures for Glaucoma Detection Using Retinal Fundus Images  
**Architecture:** DenseNet121  
**Dataset:** ACRIMA, binary classification: Glaucoma vs Normal

This notebook mirrors the ResNet50 experiment structure so the comparison is fair: same dataset split style, preprocessing pipeline, augmentation, metrics, visualizations, model saving, inference demo, and 5-fold cross-validation.

## Requirements

- Python 3.10+
- TensorFlow 2.x
- NumPy
- Pandas
- Matplotlib
- Seaborn
- Scikit-learn
- Pillow

## 1. Imports

In [ ]:
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from PIL import Image, ImageFile
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import callbacks, layers, models, optimizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

ImageFile.LOAD_TRUNCATED_IMAGES = True

print('TensorFlow version:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 2. Project Configuration

DenseNet121 is used with ImageNet transfer learning. Transfer learning helps because medical datasets are usually much smaller than ImageNet, while the pretrained backbone already captures useful low-level visual features such as edges, color transitions, vessels, and texture patterns.

In [ ]:
# Dataset Path
ACRIMA_ROOT = Path("../dataset")
ACRIMA_IMAGES_DIR = ACRIMA_ROOT / "Images"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
SEED = 42

OUTPUT_DIR = Path("../results/densenet121")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

assert ACRIMA_ROOT.exists(), f'ACRIMA root not found: {ACRIMA_ROOT}'
assert ACRIMA_IMAGES_DIR.exists(), f'Images folder not found: {ACRIMA_IMAGES_DIR}'

print('ACRIMA root:', ACRIMA_ROOT)
print('Images folder:', ACRIMA_IMAGES_DIR)
print('Output directory:', OUTPUT_DIR)

## 3. Build ACRIMA Dataframe

According to the ACRIMA readme, filenames containing `_g_` are glaucomatous/pathological images. Filenames without `_g_` are normal images.

## Dataset

This project uses the **ACRIMA retinal fundus dataset**.

The dataset is **not included** in this repository because of licensing and distribution restrictions.

After downloading the dataset, place it in:

```
dataset/
└── Images/
```

In [ ]:
image_files = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']:
    image_files.extend(ACRIMA_IMAGES_DIR.glob(ext))
    image_files.extend(ACRIMA_IMAGES_DIR.glob(ext.upper()))

rows = []
for image_path in image_files:
    filename = image_path.name.lower()
    label = 1 if '_g_' in filename else 0
    rows.append(
        {
            'image_path': str(image_path),
            'image_name': image_path.name,
            'label': label,
            'class_name': 'Glaucoma' if label == 1 else 'Normal',
        }
    )

df = pd.DataFrame(rows).sort_values('image_name').reset_index(drop=True)
if df.empty:
    raise ValueError('No images found. Check ACRIMA_IMAGES_DIR.')

# Fast Drive-friendly validation: check paths exist without opening every large image.
df = df[df['image_path'].apply(lambda p: Path(p).exists())].reset_index(drop=True)

print('Total images:', len(df))
print(df['class_name'].value_counts())
display(df.head())
display(df.tail())

## Running the Notebook

1. Download the ACRIMA dataset.
2. Create the following folder structure:

dataset/
└── Images/

3. Install dependencies from `requirements.txt`.
4. Run all cells sequentially.

## 4. Train, Validation, and Test Split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=SEED,
)

print('Train:', train_df.shape, train_df['class_name'].value_counts().to_dict())
print('Validation:', val_df.shape, val_df['class_name'].value_counts().to_dict())
print('Test:', test_df.shape, test_df['class_name'].value_counts().to_dict())

classes = np.array([0, 1])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_df['label'].values)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}
print('Class weights:', class_weight_dict)

## 5. Preprocessing and Augmentation

Preprocessing matters because DenseNet121 with ImageNet weights expects inputs normalized with the DenseNet-specific `preprocess_input`. Augmentation helps reduce overfitting by exposing the model to realistic variations in fundus image orientation, zoom, and illumination.

In [ ]:
class RandomBrightness(layers.Layer):
    def __init__(self, max_delta=0.12, **kwargs):
        super().__init__(**kwargs)
        self.max_delta = max_delta

    def call(self, inputs, training=None):
        if training:
            return tf.image.random_brightness(inputs, max_delta=self.max_delta)
        return inputs

    def get_config(self):
        config = super().get_config()
        config.update({'max_delta': self.max_delta})
        return config

data_augmentation = tf.keras.Sequential(
    [
        layers.RandomRotation(0.08, seed=SEED),
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomZoom(height_factor=0.10, width_factor=0.10, seed=SEED),
        RandomBrightness(max_delta=0.12),
    ],
    name='data_augmentation',
)

def load_and_preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, tf.cast(label, tf.float32)

def make_dataset(dataframe, shuffle=False, augment=False):
    paths = dataframe['image_path'].values
    labels = dataframe['label'].values.astype(np.float32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        dataset = dataset.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True, augment=True)
val_ds = make_dataset(val_df, shuffle=False, augment=False)
test_ds = make_dataset(test_df, shuffle=False, augment=False)

for images, labels in train_ds.take(1):
    print('Batch image shape:', images.shape)
    print('Batch label shape:', labels.shape)

## 6. Build DenseNet121 Model

DenseNet121 is suitable for medical imaging because dense connections encourage feature reuse across layers, which can help with subtle optic-disc patterns. Overfitting is controlled using augmentation, dropout, class weighting, early stopping, and a frozen pretrained backbone during initial training.

In [ ]:
def build_densenet121_model(input_shape=(224, 224, 3), dropout_rate=0.40):
    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False

    inputs = layers.Input(shape=input_shape, name='fundus_image')
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
    x = layers.BatchNormalization(name='batch_norm')(x)
    x = layers.Dense(256, activation='relu', name='dense_256')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='glaucoma_probability')(x)
    return models.Model(inputs, outputs, name='DenseNet121_ACRIMA_Glaucoma'), base_model

model, base_model = build_densenet121_model(input_shape=IMG_SIZE + (3,))
model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 7. Training

In [ ]:
best_model_path = OUTPUT_DIR / 'best_densenet121_acrima.keras'
training_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(str(best_model_path), monitor='val_loss', save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
]

start_time = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=training_callbacks,
    class_weight=class_weight_dict,
)
training_time_seconds = time.time() - start_time
print(f'Training time: {training_time_seconds:.2f} seconds ({training_time_seconds / 60:.2f} minutes)')

## 8. Training Curves

In [ ]:
history_df = pd.DataFrame(history.history)
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history_df['accuracy'], label='Training Accuracy')
plt.plot(history_df['val_accuracy'], label='Validation Accuracy')
plt.title('DenseNet121 Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(history_df['loss'], label='Training Loss')
plt.plot(history_df['val_loss'], label='Validation Loss')
plt.title('DenseNet121 Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Crossentropy Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Test Evaluation

In [ ]:
y_true = test_df['label'].values.astype(int)
test_start = time.time()
y_prob = model.predict(test_ds).ravel()
total_inference_time = time.time() - test_start
y_pred = (y_prob >= 0.5).astype(int)
inference_time_per_image = total_inference_time / len(test_df)

test_accuracy = accuracy_score(y_true, y_pred)
test_precision = precision_score(y_true, y_pred, zero_division=0)
test_recall = recall_score(y_true, y_pred, zero_division=0)
test_f1 = f1_score(y_true, y_pred, zero_division=0)
test_auc = roc_auc_score(y_true, y_prob)

print(f'Test Accuracy:  {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall/Sensitivity: {test_recall:.4f}')
print(f'Test F1-score:  {test_f1:.4f}')
print(f'ROC-AUC Score:  {test_auc:.4f}')
print(f'Inference time per image: {inference_time_per_image:.6f} seconds')
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=['Normal', 'Glaucoma'], zero_division=0))

## 10. Confusion Matrix and ROC Curve

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Glaucoma'], yticklabels=['Normal', 'Glaucoma'])
plt.title('Confusion Matrix - DenseNet121')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'DenseNet121 ROC curve (AUC = {test_auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random classifier')
plt.title('ROC Curve - DenseNet121 Glaucoma Detection')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

## 11. Save Model and Measure Size

In [ ]:
h5_model_path = OUTPUT_DIR / 'densenet121_acrima_model.h5'
saved_model_path = OUTPUT_DIR / 'densenet121_acrima_saved_model'
model.save(str(h5_model_path))
try:
    model.export(str(saved_model_path))
except AttributeError:
    model.save(str(saved_model_path))

h5_size_mb = h5_model_path.stat().st_size / (1024 * 1024)
saved_model_size_mb = sum(p.stat().st_size for p in saved_model_path.rglob('*') if p.is_file()) / (1024 * 1024)
print('Saved H5 model:', h5_model_path)
print('SavedModel directory:', saved_model_path)
print(f'H5 model size: {h5_size_mb:.2f} MB')
print(f'SavedModel size: {saved_model_size_mb:.2f} MB')

## 12. Random Prediction Demo

In [ ]:
def preprocess_single_image(image_path):
    image = Image.open(image_path).convert('RGB')
    display_image = image.copy()
    image = image.resize(IMG_SIZE)
    image_array = np.array(image).astype(np.float32)
    image_array = preprocess_input(image_array)
    image_array = np.expand_dims(image_array, axis=0)
    return image_array, display_image

sample_rows = test_df.sample(6, random_state=random.randint(0, 10_000))
plt.figure(figsize=(14, 8))
for i, (_, sample_row) in enumerate(sample_rows.iterrows(), start=1):
    sample_array, sample_display = preprocess_single_image(sample_row['image_path'])
    prob = float(model.predict(sample_array, verbose=0)[0][0])
    pred_label = 'Glaucoma' if prob >= 0.5 else 'Normal'
    confidence = prob if pred_label == 'Glaucoma' else 1.0 - prob
    if confidence < 0.70:
        confidence_level = 'Moderate'
    elif confidence < 0.90:
        confidence_level = 'High'
    else:
        confidence_level = 'Very High'
    plt.subplot(2, 3, i)
    plt.imshow(sample_display)
    plt.axis('off')
    plt.title(f"True: {sample_row['class_name']}\nPred: {pred_label}\nConf: {confidence:.2%} ({confidence_level})", fontsize=10)
plt.tight_layout()
plt.show()

## 13. Five-Fold Stratified Cross-Validation

Cross-validation provides a more reliable estimate than a single split. It is especially useful in biomedical datasets where performance can vary with the test split.

In [ ]:
N_SPLITS = 5
CV_EPOCHS = 20
cv_results = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (trainval_idx, test_idx) in enumerate(skf.split(df, df['label']), start=1):
    print(f'\n========== Fold {fold}/{N_SPLITS} ==========')
    trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
    fold_test_df = df.iloc[test_idx].reset_index(drop=True)
    fold_train_df, fold_val_df = train_test_split(
        trainval_df,
        test_size=0.15,
        stratify=trainval_df['label'],
        random_state=SEED + fold,
    )

    fold_train_ds = make_dataset(fold_train_df, shuffle=True, augment=True)
    fold_val_ds = make_dataset(fold_val_df, shuffle=False, augment=False)
    fold_test_ds = make_dataset(fold_test_df, shuffle=False, augment=False)

    fold_weights = compute_class_weight(class_weight='balanced', classes=classes, y=fold_train_df['label'].values)
    fold_class_weight = {int(c): float(w) for c, w in zip(classes, fold_weights)}

    fold_model, _ = build_densenet121_model(input_shape=IMG_SIZE + (3,))
    fold_model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    fold_callbacks = [
        callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    ]

    fold_start = time.time()
    fold_model.fit(
        fold_train_ds,
        validation_data=fold_val_ds,
        epochs=CV_EPOCHS,
        callbacks=fold_callbacks,
        class_weight=fold_class_weight,
        verbose=1,
    )
    fold_training_time = time.time() - fold_start

    y_true_fold = fold_test_df['label'].values.astype(int)
    y_prob_fold = fold_model.predict(fold_test_ds).ravel()
    y_pred_fold = (y_prob_fold >= 0.5).astype(int)

    cv_results.append(
        {
            'fold': fold,
            'accuracy': accuracy_score(y_true_fold, y_pred_fold),
            'precision': precision_score(y_true_fold, y_pred_fold, zero_division=0),
            'recall_sensitivity': recall_score(y_true_fold, y_pred_fold, zero_division=0),
            'f1_score': f1_score(y_true_fold, y_pred_fold, zero_division=0),
            'roc_auc': roc_auc_score(y_true_fold, y_prob_fold),
            'training_time_seconds': fold_training_time,
            'test_normal_count': int((y_true_fold == 0).sum()),
            'test_glaucoma_count': int((y_true_fold == 1).sum()),
        }
    )
    print(classification_report(y_true_fold, y_pred_fold, target_names=['Normal', 'Glaucoma'], zero_division=0))

cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df)
summary_df = cv_results_df[['accuracy', 'precision', 'recall_sensitivity', 'f1_score', 'roc_auc']].agg(['mean', 'std'])
display(summary_df)
print('5-Fold Cross-Validation Summary')
print(f"Accuracy:    {cv_results_df['accuracy'].mean():.4f} ± {cv_results_df['accuracy'].std():.4f}")
print(f"Precision:   {cv_results_df['precision'].mean():.4f} ± {cv_results_df['precision'].std():.4f}")
print(f"Sensitivity: {cv_results_df['recall_sensitivity'].mean():.4f} ± {cv_results_df['recall_sensitivity'].std():.4f}")
print(f"F1-score:    {cv_results_df['f1_score'].mean():.4f} ± {cv_results_df['f1_score'].std():.4f}")
print(f"ROC-AUC:     {cv_results_df['roc_auc'].mean():.4f} ± {cv_results_df['roc_auc'].std():.4f}")

## 14. Experiment Summary

In [ ]:
summary = {
    'Architecture': 'DenseNet121 transfer learning',
    'Dataset': 'ACRIMA optic-disc cropped fundus images',
    'Train images': len(train_df),
    'Validation images': len(val_df),
    'Test images': len(test_df),
    'Test accuracy': round(test_accuracy, 4),
    'Test precision': round(test_precision, 4),
    'Test recall/sensitivity': round(test_recall, 4),
    'Test F1-score': round(test_f1, 4),
    'ROC-AUC': round(test_auc, 4),
    'Training time seconds': round(training_time_seconds, 2),
    'Inference time per image seconds': round(inference_time_per_image, 6),
    'H5 model size MB': round(h5_size_mb, 2),
    'SavedModel size MB': round(saved_model_size_mb, 2),
}
display(pd.DataFrame([summary]))
print('DenseNet121 is a strong medical-imaging comparison model for ResNet50 because dense connections improve feature reuse.')
print('For Raspberry Pi deployment, DenseNet121 may still be heavier than MobileNetV2, so MobileNetV2 should be tested as the lightweight model next.')

This notebook is released for academic and research purposes.